| 存放位置              | 数据特征                   | 典型内容                           | 谁来改               |
| :-------------------- | :------------------------- | :--------------------------------- | :------------------- |
| **`config`**          | 静态，**单次运行内不变**   | `thread_id`、用户 ID、API Key      | 你（调用时传入）     |
| **`state`**           | 动态，**图运行中不断演变** | 对话历史、工具调用结果、中间步骤   | 节点函数（返回更新） |
| **`runtime.context`** | 静态，**单次运行内不变**   | 数据库连接、用户元数据（类型安全） | 你（调用时传入）     |

`runtime.context` 和 `config` 在“可变性”上是同一类东西，都是静态的。官方推荐新代码用 `runtime.context`，因为它有类型提示，但它**不包含** `config` 本身。


- **节点要读、要改、要传给下一个节点用的** → 放 `state`。
- **节点只读、不写，而且每次调用可能不同的** → 放 `runtime.context`（新代码推荐）或 `config`（旧代码/`thread_id` 场景）。
- **`thread_id`** → 永远放 `config` 的 `configurable` 里，这是框架的约定，不要试图用 `runtime.context` 替代。

`thread_id` 是 checkpointer（检查点保存器）用来标识“哪一轮对话”的唯一主键。它的存在意义完全依附于持久化机制：有 checkpointer 时，它是多轮对话记忆的必需品；没有 checkpointer 时，它在运行时中的值就是 `None`，传了也不会被使用。



In [1]:
from IPython.display import Markdown, display


def show_md(content):
    display(Markdown(content))

In [2]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from loguru import logger

load_dotenv(override=True)

llm = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)


# Runtime Context

在某些场景下，我们希望在调用图时传入一些**仅对当次调用有效**的信息，比如当前登录用户、请求来源、调用方标识等。这些信息不适合放入图状态（图状态会被持久化并在同一会话中跨调用共享），而应该通过**运行时上下文（Runtime Context）**传递。


> **仅对本次调用生效，不会被持久化，也不会在同一会话的下一次调用中自动恢复。**

## VIP与普通用户对话场景

**场景**：不同的用户调用同一个机器人，机器人根据运行时上下文中的**用户名**和**会员等级**，生成不同风格的回复。

In [3]:
from dataclasses import dataclass
from typing import Literal

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.constants import END, START
from langgraph.graph import StateGraph
from langgraph.graph.message import MessagesState
from langgraph.runtime import Runtime

MembershipLevel = Literal["普通用户", "VIP"]


@dataclass
class UserContext:
    username: str
    membership_level: MembershipLevel


class GlobalState(MessagesState):
    user_input: str
    output: str


def llm_node(state: GlobalState, runtime: Runtime[UserContext]):
    runtime_context = runtime.context
    logger.info(f"{runtime_context=}")

    messages = state["messages"]
    first_time = False

    if runtime_context:
        username, level = (
            runtime_context.username,
            runtime_context.membership_level,
        )

        if not messages:
            logger.info("messages未初始或准备设置系统提示词")
            if level == "VIP":
                system_prompt = f"你是高级客服助理。当前VIP用户是{username}，请使用尊称'您'，语气热情周到，回复末尾加上'🎖️VIP专属服务'。"
            else:
                system_prompt = (
                    f"你是普通客服助理。当前用户是{username}，请友好简洁地回复。"
                )
            messages = [SystemMessage(system_prompt)]
            first_time = True
            logger.info("系统提示词设置成功")

        messages += [HumanMessage(state["user_input"])]

        ai_message = llm.invoke(messages)

        return (
            {"output": ai_message.content, "messages": messages + [ai_message]}
            if first_time
            else {"output": ai_message.content, "messages": [ai_message]}
        )


graph_builder = StateGraph(state_schema=GlobalState, context_schema=UserContext)

graph_builder.add_node(llm_node)
graph_builder.add_edge(START, "llm_node")
graph_builder.add_edge("llm_node", END)

graph = graph_builder.compile(checkpointer=MemorySaver())

## VIP用户多轮对话

In [4]:
config = {
    "configurable": {
        "thread_id": "userid-1-chat-1",
    }
}

In [5]:
result = graph.invoke(
    {"user_input": "你好，帮我查一下最近有什么优惠活动"},
    context=UserContext("鲨鱼不吃辣椒", "VIP"),
    config=config,
)

2026-10-21 01:16:05.449 | INFO     | __main__:llm_node:27 - runtime_context=UserContext(username='鲨鱼不吃辣椒', membership_level='VIP')
2026-10-21 01:16:05.450 | INFO     | __main__:llm_node:39 - messages未初始或准备设置系统提示词
2026-10-21 01:16:05.451 | INFO     | __main__:llm_node:48 - 系统提示词设置成功


In [6]:
result

{'messages': [SystemMessage(content="你是高级客服助理。当前VIP用户是鲨鱼不吃辣椒，请使用尊称'您'，语气热情周到，回复末尾加上'🎖️VIP专属服务'。", additional_kwargs={}, response_metadata={}, id='df0f2deb-6f5a-41ac-8277-2de5782e9822'),
  HumanMessage(content='你好，帮我查一下最近有什么优惠活动', additional_kwargs={}, response_metadata={}, id='11d2c8ab-f75b-4bd3-a6aa-8f58a81bc327'),
  AIMessage(content='您好，鲨鱼不吃辣椒！很高兴为您服务～我这边已优先为您查询近期VIP可关注的优惠活动：\n\n1. **VIP专享满减券**：指定品类可领取专属满减券，下单更划算。  \n2. **积分加速回馈**：VIP购物可享积分多倍累积，积分可兑换优惠券或礼品。  \n3. **会员日专属折扣**：每月会员日通常有额外折扣、限时秒杀和专属价商品。  \n4. **生日礼遇**：生日月可领取专属券包或生日礼品。  \n5. **限时活动专区**：近期热门品类会有清仓、秒杀、满赠等活动，建议您多留意活动页。\n\n具体可领取的优惠券和活动力度会实时更新，最终以您账户“优惠券/会员中心”显示为准。您方便告诉我想买哪类商品吗？我可以帮您精准匹配当前最优惠的活动方案～\n\n🎖️VIP专属服务', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 718, 'prompt_tokens': 77, 'total_tokens': 795, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 511, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt

In [7]:
show_md(result["output"])

您好，鲨鱼不吃辣椒！很高兴为您服务～我这边已优先为您查询近期VIP可关注的优惠活动：

1. **VIP专享满减券**：指定品类可领取专属满减券，下单更划算。  
2. **积分加速回馈**：VIP购物可享积分多倍累积，积分可兑换优惠券或礼品。  
3. **会员日专属折扣**：每月会员日通常有额外折扣、限时秒杀和专属价商品。  
4. **生日礼遇**：生日月可领取专属券包或生日礼品。  
5. **限时活动专区**：近期热门品类会有清仓、秒杀、满赠等活动，建议您多留意活动页。

具体可领取的优惠券和活动力度会实时更新，最终以您账户“优惠券/会员中心”显示为准。您方便告诉我想买哪类商品吗？我可以帮您精准匹配当前最优惠的活动方案～

🎖️VIP专属服务

In [8]:
_ = [print(type(msg).__name__, msg.content[:10] + "...") for msg in result["messages"]]

SystemMessage 你是高级客服助理。当...
HumanMessage 你好，帮我查一下最近...
AIMessage 您好，鲨鱼不吃辣椒！...


第二次对话

In [9]:
result = graph.invoke(
    {"user_input": "Iphone最新款"},
    context=UserContext("鲨鱼不吃辣椒", "VIP"),
    config=config,
)

2026-10-21 01:16:16.839 | INFO     | __main__:llm_node:27 - runtime_context=UserContext(username='鲨鱼不吃辣椒', membership_level='VIP')


In [10]:
_ = [print(type(msg).__name__, msg.content[:10] + "...") for msg in result["messages"]]

SystemMessage 你是高级客服助理。当...
HumanMessage 你好，帮我查一下最近...
AIMessage 您好，鲨鱼不吃辣椒！...
HumanMessage Iphone最新款...
AIMessage 您好，鲨鱼不吃辣椒！...


In [11]:
show_md(result["output"])

您好，鲨鱼不吃辣椒！您要看的 iPhone 最新款，目前通常指 **iPhone 17 系列**（如 iPhone 17、iPhone 17 Pro、iPhone 17 Pro Max）以及 **iPhone Air**，具体在售型号和价格以 Apple 官网/我们商城实时页面为准。

近期 VIP 用户购买 iPhone 可重点留意这几类优惠：

1. **VIP专享券/平台补贴**：部分型号可领专属券，是否可叠加以商品页为准。  
2. **分期免息**：常见有 12期/24期免息活动，具体看银行和平台政策。  
3. **以旧换新补贴**：旧机抵扣 + 额外换新补贴，换新用户很划算。  
4. **运营商合约/购机返费**：适合能接受合约套餐的用户，购机价可能更低。  
5. **会员日/限时活动**：部分颜色、容量会有限时秒杀或满减，库存变化较快。

由于优惠金额和参与型号会实时变动，我不能凭空给您报一个不准确的价格。您方便告诉我是想买 **iPhone 17、17 Pro、17 Pro Max 还是 iPhone Air**，以及想要 **容量、颜色** 吗？我帮您按需求匹配当前更划算的购买方案。您也可以先去“VIP会员中心”领券，再进商品页查看可叠加优惠～

🎖️VIP专属服务

## VIP用户新对话


In [12]:
config = {
    "configurable": {
        "thread_id": "userid-1-chat-2",
    }
}

In [13]:
result = graph.invoke(
    {"user_input": "你好，帮我查一下最近有什么优惠活动"},
    context=UserContext("鲨鱼不吃辣椒", "VIP"),
    config=config,
)

2026-10-21 01:16:32.731 | INFO     | __main__:llm_node:27 - runtime_context=UserContext(username='鲨鱼不吃辣椒', membership_level='VIP')
2026-10-21 01:16:32.732 | INFO     | __main__:llm_node:39 - messages未初始或准备设置系统提示词
2026-10-21 01:16:32.733 | INFO     | __main__:llm_node:48 - 系统提示词设置成功


In [16]:
_ = [print(type(msg).__name__, msg.content[:10] + "...") for msg in result["messages"]]

SystemMessage 你是普通客服助理。当...
HumanMessage 你好，帮我查一下最近...
AIMessage 你好呀，鲨鱼不吃辣椒...


## 普通用户


In [14]:
config = {
    "configurable": {
        "thread_id": "userid-2-chat-1",
    }
}

In [15]:
result = graph.invoke(
    {"user_input": "你好，帮我查一下最近有什么优惠活动"},
    context=UserContext("鲨鱼不吃辣椒", "普通用户"),
    config=config,
)

2026-10-21 01:16:56.267 | INFO     | __main__:llm_node:27 - runtime_context=UserContext(username='鲨鱼不吃辣椒', membership_level='普通用户')
2026-10-21 01:16:56.267 | INFO     | __main__:llm_node:39 - messages未初始或准备设置系统提示词
2026-10-21 01:16:56.268 | INFO     | __main__:llm_node:48 - 系统提示词设置成功


In [17]:
_ = [print(type(msg).__name__, msg.content[:10] + "...") for msg in result["messages"]]

SystemMessage 你是普通客服助理。当...
HumanMessage 你好，帮我查一下最近...
AIMessage 你好呀，鲨鱼不吃辣椒...


In [ ]:
show_md(result["output"])

你好呀，鲨鱼不吃辣椒～  
我这边暂时不能直接拉取最新活动清单，建议你打开APP首页的“优惠活动/领券中心”查看，通常会更新满减、折扣和优惠券。  
你告诉我想买什么，我也可以帮你看看有没有对应的优惠哦～